### LongWanjuan: Towards Systematic Measurement for Long Text Quality

#### Import necessary libraries and load the pre-trained language model
In this step, we import libraries from NLTK and HuggingFace's transformers. We load the GPT-2 tokenizer and model to measure the coherence and complexity of long texts.

In [4]:
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
nltk.download('punkt')

# Load pre-trained language model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

#### Define metrics for evaluating text quality
Below, we define three key metrics to evaluate long text quality: coherence, cohesion, and complexity.
- **Coherence:** Measures how well sentences and paragraphs flow together based on GPT-2's predictions over different context windows.
- **Cohesion:** Quantifies the presence of connectives and pronouns that help bind sentences together.
- **Complexity:** Assesses the lexical richness and sentence structure variety using TTR and average paragraph length.

In [36]:
# Function to measure coherence
def coherence_metrics(text, window_size=4096):
    # Tokenize text
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids
    
    # Split the text into smaller contexts (short and long contexts)
    short_context = input_ids[:, :window_size // 2]
    long_context = input_ids
    
    # Predict on short and long contexts
    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
        outputs_long = model(long_context, labels=long_context)
    
    # Calculate accuracy and log-likelihood difference
    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    coherence_acc_long = (outputs_long.logits.argmax(-1) == long_context).float().mean().item()
    
    coherence_diff = (coherence_acc_long - coherence_acc_short) / coherence_acc_long
    return coherence_acc_long, coherence_acc_short, coherence_diff

# Function to measure cohesion (connectives and pronouns)
def cohesion_metrics(text):
    connectives = ['but', 'however', 'therefore', 'yet', 'since', 'because', 'although']  # Add more connectives
    pronouns = ['he', 'she', 'it', 'they', 'we', 'I', 'you']  # Add more pronouns

    sentences = nltk.sent_tokenize(text)
    num_connectives = sum(1 for word in nltk.word_tokenize(text.lower()) if word in connectives)
    num_pronouns = sum(1 for word in nltk.word_tokenize(text.lower()) if word in pronouns)
    
    cohesion_conn = num_connectives / len(sentences)
    cohesion_pron = num_pronouns / len(sentences)
    
    return cohesion_conn, cohesion_pron

# Function to measure complexity (TTR and average paragraph length)
def complexity_metrics(text):
    words = nltk.word_tokenize(text)
    unique_words = set(words)
    ttr = len(unique_words) / len(words)
    
    paragraphs = text.split('\n\n')
    avg_para_length = sum(len(nltk.word_tokenize(para)) for para in paragraphs) / len(paragraphs)
    
    return ttr, avg_para_length

#### Explanation of Metrics
**Coherence - Long Acc, Short Acc, Diff:**
- **Long Acc:** Accuracy of predicting the next token using a longer context. Higher accuracy suggests better coherence.
- **Short Acc:** Accuracy of predicting the next token using a shorter context. A lower value than Long Acc indicates that a longer context helps understanding.
- **Diff:** The difference between long and short accuracies. A larger positive value indicates higher coherence.

**Cohesion - Connectives, Pronouns:**
- **Connectives:** Measures the usage of transitional words like "however" or "therefore." Higher values indicate better linkage between sentences.
- **Pronouns:** Counts the use of pronouns to maintain referential cohesion across sentences.

**Complexity - TTR, Average Paragraph Length:**
- **TTR (Type-Token Ratio):** Measures lexical diversity by comparing unique word types to the total number of words. Higher values suggest more complex vocabulary.
- **Average Paragraph Length:** Average number of words per paragraph, giving insight into how dense or expansive the writing is.

#### Evaluation of a sample text
Let's test the system on a sample text and evaluate its quality based on coherence, cohesion, and complexity.

In [39]:
# Sample text to evaluate
sample_text = '''
Technology has advanced rapidly over the last few decades. This progress has led to significant changes in various industries. However, there are concerns about the societal impact of these advancements.
In education, technology has transformed the way students learn. Online resources have made information more accessible, but they also raise questions about the quality of education. Although students can now learn at their own pace, some argue that the lack of interaction with teachers could be detrimental.
Moreover, automation is another area where technology has raised concerns. While automation has increased efficiency in manufacturing and services, there are fears that it could lead to job losses. Governments and policymakers need to address these issues to ensure a smooth transition to a more automated future.'''

# Calculate the metrics for the sample text
coh_acc_long, coh_acc_short, coh_diff = coherence_metrics(sample_text)
coh_conn, coh_pron = cohesion_metrics(sample_text)
ttr, avg_para_length = complexity_metrics(sample_text)

# Display results
print(f"Coherence - Long Acc: {coh_acc_long}, Short Acc: {coh_acc_short}, Diff: {coh_diff}")
print(f"Cohesion - Connectives per sentence: {coh_conn}, Pronouns per sentence: {coh_pron}")
print(f"Complexity - TTR: {ttr}, Avg. Paragraph Length: {avg_para_length}")

Coherence - Long Acc: 0.16111093759536743, Short Acc: 0.10117185115814209, Diff: 0.37250107859072845
Cohesion - Connectives per sentence: 0.1111111111111111, Pronouns per sentence: 0.4444444444444444
Complexity - TTR: 0.43548387096774194, Avg. Paragraph Length: 52.0
